# Walmart M5 Data Profiling

## Business Objective

Before finalizing the ETL and forecasting pipeline, all source data should be inspected to understand its **structure, quality, scale, and relationships**.

This notebook profiles both:

- the **Walmart M5 source files** stored in `data/raw`, and
- the **external datasets already created by the extraction scripts** and stored in `data/external`.

The external files are produced by:

- `etl/extract/extract_fred.py`
- `etl/extract/extract_holidays.py`
- `etl/extract/extract_weather.py`

The purpose of this notebook is to determine what the **Transform** layer actually needs to do before the data is loaded into PostgreSQL.

This notebook focuses on:

- Dataset dimensions
- Column names and structure
- Data types
- Required columns
- Memory usage
- Missing values
- Duplicate records
- Business-key uniqueness
- Identifier cardinality
- Date coverage
- Basic validity checks
- Dataset relationships and join keys
- ETL transformation requirements

> This notebook is for **data profiling**, not deep exploratory analysis. Trends, seasonality, demand behavior, price-demand relationships, weather effects, and economic relationships belong in the next EDA notebook.


## 1. Import Libraries

`Path` is used to build reusable project file paths, while pandas is used to load and inspect each dataset.


In [1]:
from pathlib import Path

import pandas as pd


## 2. Define Project Paths

The project contains two source-data locations:

- `data/raw` contains the original Walmart M5 competition files.
- `data/external` contains CSV files produced by the external-data extraction scripts.

The extraction scripts have already run before this notebook, so this notebook reads their **CSV outputs** rather than calling the APIs again.


In [2]:
PROJECT_ROOT = Path("..")

RAW_DIR = PROJECT_ROOT / "data" / "raw"
EXTERNAL_DIR = PROJECT_ROOT / "data" / "external"

# Walmart M5 source files
calendar_path = RAW_DIR / "calendar.csv"
prices_path = RAW_DIR / "sell_prices.csv"
sales_validation_path = RAW_DIR / "sales_train_validation.csv"
sales_evaluation_path = RAW_DIR / "sales_train_evaluation.csv"
submission_path = RAW_DIR / "sample_submission.csv"

# External files created by etl/extract/*.py
fred_path = EXTERNAL_DIR / "fred_economic_data.csv"
holidays_path = EXTERNAL_DIR / "us_holidays.csv"
weather_path = EXTERNAL_DIR / "weather_history.csv"


## 3. Confirm Source Files Exist

Before loading the data, confirm that every expected source file exists at the path the notebook will use.

If any result is `False`, the file path or extraction output should be checked before continuing.


In [3]:
source_paths = {
    "calendar": calendar_path,
    "prices": prices_path,
    "sales_validation": sales_validation_path,
    "sales_evaluation": sales_evaluation_path,
    "submission": submission_path,
    "fred": fred_path,
    "holidays": holidays_path,
    "weather": weather_path,
}

for name, path in source_paths.items():
    print(f"{name.upper():18} Exists: {path.exists()} | {path}")


CALENDAR           Exists: True | ..\data\raw\calendar.csv
PRICES             Exists: True | ..\data\raw\sell_prices.csv
SALES_VALIDATION   Exists: True | ..\data\raw\sales_train_validation.csv
SALES_EVALUATION   Exists: True | ..\data\raw\sales_train_evaluation.csv
SUBMISSION         Exists: True | ..\data\raw\sample_submission.csv
FRED               Exists: True | ..\data\external\fred_economic_data.csv
HOLIDAYS           Exists: True | ..\data\external\us_holidays.csv
WEATHER            Exists: True | ..\data\external\weather_history.csv


## 4. Load Source Datasets

Each CSV is loaded into a pandas DataFrame for profiling.

The M5 files were obtained directly from the competition data. The FRED, holiday, and weather files are the saved outputs of the project's extraction scripts.


In [4]:
calendar = pd.read_csv(calendar_path)
prices = pd.read_csv(prices_path)
sales = pd.read_csv(sales_validation_path)
evaluation = pd.read_csv(sales_evaluation_path)
submission = pd.read_csv(submission_path)

fred = pd.read_csv(fred_path)
holidays = pd.read_csv(holidays_path)
weather = pd.read_csv(weather_path)

datasets = {
    "calendar": calendar,
    "prices": prices,
    "sales_validation": sales,
    "sales_evaluation": evaluation,
    "submission": submission,
    "fred": fred,
    "holidays": holidays,
    "weather": weather,
}


## 5. Dataset Inventory

The first profiling step is to confirm the size of every source dataset.

For the Walmart M5 files, the expected sizes are:

- Calendar: **1,969 rows × 14 columns**
- Sell prices: **6,841,121 rows × 4 columns**
- Sales validation: **30,490 rows × 1,919 columns**
- Sales evaluation: **30,490 rows × 1,947 columns**
- Sample submission: **60,980 rows × 29 columns**

The external dataset sizes are printed dynamically because they depend on the extraction output.


In [5]:
inventory = []

for name, df in datasets.items():
    inventory.append({
        "dataset": name,
        "rows": df.shape[0],
        "columns": df.shape[1],
    })

inventory_df = pd.DataFrame(inventory)
inventory_df


,dataset,rows,columns
0,calendar,1969,14
1,prices,6841121,4
2,sales_validation,30490,1919
3,sales_evaluation,30490,1947
4,submission,60980,29
5,fred,198,3
6,holidays,65,2
7,weather,5907,7


### Profiling Note

The M5 sales files are stored in a **wide format**, with one column per historical day.

That format is useful for the original competition files but is not appropriate for the PostgreSQL sales fact table. The sales Transform step therefore needs to reshape the daily `d_*` columns from **wide format to long format**.


## 6. Preview Source Data

Previewing the first few records helps confirm what each source contains and makes the schemas visible before transformation logic is written.


### Calendar Dataset


In [6]:
calendar.head()


,date,wm_yr_wk,weekday,wday,month,year,d,event_name_1,event_type_1,event_name_2,event_type_2,snap_CA,snap_TX,snap_WI
0,2011-01-29,11101,Saturday,1,1,2011,d_1,NaN,NaN,NaN,NaN,0,0,0
1,2011-01-30,11101,Sunday,2,1,2011,d_2,NaN,NaN,NaN,NaN,0,0,0
2,2011-01-31,11101,Monday,3,1,2011,d_3,NaN,NaN,NaN,NaN,0,0,0
3,2011-02-01,11101,Tuesday,4,2,2011,d_4,NaN,NaN,NaN,NaN,1,1,0
4,2011-02-02,11101,Wednesday,5,2,2011,d_5,NaN,NaN,NaN,NaN,1,0,1


### Sell Prices Dataset


In [7]:
prices.head()


,store_id,item_id,wm_yr_wk,sell_price
0,CA_1,HOBBIES_1_001,11325,9.58
1,CA_1,HOBBIES_1_001,11326,9.58
2,CA_1,HOBBIES_1_001,11327,8.26
3,CA_1,HOBBIES_1_001,11328,8.26
4,CA_1,HOBBIES_1_001,11329,8.26


### Sales Validation Dataset


In [8]:
sales.head()


,id,item_id,dept_id,cat_id,store_id,state_id,d_1,d_2,d_3,d_4,...,d_1904,d_1905,d_1906,d_1907,d_1908,d_1909,d_1910,d_1911,d_1912,d_1913
0,HOBBIES_1_001_CA_1_validation,HOBBIES_1_001,HOBBIES_1,HOBBIES,CA_1,CA,0,0,0,0,...,1,3,0,1,1,1,3,0,1,1
1,HOBBIES_1_002_CA_1_validation,HOBBIES_1_002,HOBBIES_1,HOBBIES,CA_1,CA,0,0,0,0,...,0,0,0,0,0,1,0,0,0,0
2,HOBBIES_1_003_CA_1_validation,HOBBIES_1_003,HOBBIES_1,HOBBIES,CA_1,CA,0,0,0,0,...,2,1,2,1,1,1,0,1,1,1
3,HOBBIES_1_004_CA_1_validation,HOBBIES_1_004,HOBBIES_1,HOBBIES,CA_1,CA,0,0,0,0,...,1,0,5,4,1,0,1,3,7,2
4,HOBBIES_1_005_CA_1_validation,HOBBIES_1_005,HOBBIES_1,HOBBIES,CA_1,CA,0,0,0,0,...,2,1,1,0,1,1,2,2,2,4


### Sales Evaluation Dataset


In [9]:
evaluation.head()


,id,item_id,dept_id,cat_id,store_id,state_id,d_1,d_2,d_3,d_4,...,d_1932,d_1933,d_1934,d_1935,d_1936,d_1937,d_1938,d_1939,d_1940,d_1941
0,HOBBIES_1_001_CA_1_evaluation,HOBBIES_1_001,HOBBIES_1,HOBBIES,CA_1,CA,0,0,0,0,...,2,4,0,0,0,0,3,3,0,1
1,HOBBIES_1_002_CA_1_evaluation,HOBBIES_1_002,HOBBIES_1,HOBBIES,CA_1,CA,0,0,0,0,...,0,1,2,1,1,0,0,0,0,0
2,HOBBIES_1_003_CA_1_evaluation,HOBBIES_1_003,HOBBIES_1,HOBBIES,CA_1,CA,0,0,0,0,...,1,0,2,0,0,0,2,3,0,1
3,HOBBIES_1_004_CA_1_evaluation,HOBBIES_1_004,HOBBIES_1,HOBBIES,CA_1,CA,0,0,0,0,...,1,1,0,4,0,1,3,0,2,6
4,HOBBIES_1_005_CA_1_evaluation,HOBBIES_1_005,HOBBIES_1,HOBBIES,CA_1,CA,0,0,0,0,...,0,0,0,2,1,0,0,2,1,0


### Sample Submission Dataset


In [10]:
submission.head()


,id,F1,F2,F3,F4,F5,F6,F7,F8,F9,...,F19,F20,F21,F22,F23,F24,F25,F26,F27,F28
0,HOBBIES_1_001_CA_1_validation,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,HOBBIES_1_002_CA_1_validation,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,HOBBIES_1_003_CA_1_validation,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,HOBBIES_1_004_CA_1_validation,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,HOBBIES_1_005_CA_1_validation,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


### FRED Economic Dataset


In [11]:
fred.head()


,date,value,series_id
0,2011-01-01,221.187,CPIAUCSL
1,2011-02-01,221.898,CPIAUCSL
2,2011-03-01,223.046,CPIAUCSL
3,2011-04-01,224.093,CPIAUCSL
4,2011-05-01,224.806,CPIAUCSL


### U.S. Holidays Dataset


In [12]:
holidays.head()


,date,holiday_name
0,2011-01-01,New Year's Day
1,2011-01-17,Martin Luther King Jr. Day
2,2011-02-21,Washington's Birthday
3,2011-05-30,Memorial Day
4,2011-07-04,Independence Day


### Weather History Dataset


In [13]:
weather.head()


,time,temperature_2m_max,temperature_2m_min,precipitation_sum,snowfall_sum,wind_speed_10m_max,state_id
0,2011-01-29,18.1,6.1,0.0,0.0,10.9,CA
1,2011-01-30,13.1,7.6,1.0,0.0,16.8,CA
2,2011-01-31,18.9,4.3,0.0,0.0,11.0,CA
3,2011-02-01,17.2,4.9,0.0,0.0,10.9,CA
4,2011-02-02,16.4,2.1,0.0,0.0,8.8,CA


## 7. Column Names

Before checking data quality, inspect the available column names for each source.

This is especially important for the external extracts because their schemas come from API/extraction logic rather than from the original M5 files.


In [14]:
for name, df in datasets.items():
    print(f"\n{name.upper()}")
    print(df.columns.tolist())



CALENDAR
['date', 'wm_yr_wk', 'weekday', 'wday', 'month', 'year', 'd', 'event_name_1', 'event_type_1', 'event_name_2', 'event_type_2', 'snap_CA', 'snap_TX', 'snap_WI']

PRICES
['store_id', 'item_id', 'wm_yr_wk', 'sell_price']

SALES_VALIDATION
['id', 'item_id', 'dept_id', 'cat_id', 'store_id', 'state_id', 'd_1', 'd_2', 'd_3', 'd_4', 'd_5', 'd_6', 'd_7', 'd_8', 'd_9', 'd_10', 'd_11', 'd_12', 'd_13', 'd_14', 'd_15', 'd_16', 'd_17', 'd_18', 'd_19', 'd_20', 'd_21', 'd_22', 'd_23', 'd_24', 'd_25', 'd_26', 'd_27', 'd_28', 'd_29', 'd_30', 'd_31', 'd_32', 'd_33', 'd_34', 'd_35', 'd_36', 'd_37', 'd_38', 'd_39', 'd_40', 'd_41', 'd_42', 'd_43', 'd_44', 'd_45', 'd_46', 'd_47', 'd_48', 'd_49', 'd_50', 'd_51', 'd_52', 'd_53', 'd_54', 'd_55', 'd_56', 'd_57', 'd_58', 'd_59', 'd_60', 'd_61', 'd_62', 'd_63', 'd_64', 'd_65', 'd_66', 'd_67', 'd_68', 'd_69', 'd_70', 'd_71', 'd_72', 'd_73', 'd_74', 'd_75', 'd_76', 'd_77', 'd_78', 'd_79', 'd_80', 'd_81', 'd_82', 'd_83', 'd_84', 'd_85', 'd_86', 'd_87', 'd_88

## 8. Data-Type Inspection

Data types help identify columns that require conversion during transformation.

Because the two M5 sales files contain more than 1,900 columns, their types are summarized rather than printing every daily column individually.


In [15]:
for name, df in datasets.items():
    print(f"\n{name.upper()}")
    print(df.dtypes.value_counts())



CALENDAR
str      7
int64    7
Name: count, dtype: int64

PRICES
str        2
int64      1
float64    1
Name: count, dtype: int64

SALES_VALIDATION
int64    1913
str         6
Name: count, dtype: int64

SALES_EVALUATION
int64    1941
str         6
Name: count, dtype: int64

SUBMISSION
int64    28
str       1
Name: count, dtype: int64

FRED
str        2
float64    1
Name: count, dtype: int64

HOLIDAYS
str    2
Name: count, dtype: int64

WEATHER
float64    5
str        2
Name: count, dtype: int64


### Important M5 Column Types

Important identifier and business columns are inspected separately.


In [16]:
print("CALENDAR")
print(calendar.dtypes)

print("\nPRICES")
print(prices.dtypes)

print("\nSALES IDENTIFIERS")
print(
    sales[
        ["id", "item_id", "dept_id", "cat_id", "store_id", "state_id"]
    ].dtypes
)


CALENDAR
date              str
wm_yr_wk        int64
weekday           str
wday            int64
month           int64
year            int64
d                 str
event_name_1      str
event_type_1      str
event_name_2      str
event_type_2      str
snap_CA         int64
snap_TX         int64
snap_WI         int64
dtype: object

PRICES
store_id          str
item_id           str
wm_yr_wk        int64
sell_price    float64
dtype: object

SALES IDENTIFIERS
id          str
item_id     str
dept_id     str
cat_id      str
store_id    str
state_id    str
dtype: object


### External Dataset Types

The full schemas of FRED, holidays, and weather are small enough to inspect directly.


In [17]:
print("FRED")
print(fred.dtypes)

print("\nHOLIDAYS")
print(holidays.dtypes)

print("\nWEATHER")
print(weather.dtypes)


FRED
date             str
value        float64
series_id        str
dtype: object

HOLIDAYS
date            str
holiday_name    str
dtype: object

WEATHER
time                      str
temperature_2m_max    float64
temperature_2m_min    float64
precipitation_sum     float64
snowfall_sum          float64
wind_speed_10m_max    float64
state_id                  str
dtype: object


### Data-Type Finding

The M5 calendar `date` field is loaded as text, so the Transform layer should convert it to datetime.

The external schemas above should be reviewed the same way: date fields should become datetimes, numeric measurements should remain numeric, and identifiers should retain consistent types for joins.


## 9. Required Column Checks

Required-column checks verify that each source still contains the fields expected by downstream ETL logic.

The M5 and FRED schemas have known core fields. For holiday and weather extracts, the notebook first checks for the presence of at least one date-like field and then displays the complete schema for review.


In [18]:
required_columns = {
    "calendar": [
        "date",
        "wm_yr_wk",
        "d",
    ],
    "prices": [
        "store_id",
        "item_id",
        "wm_yr_wk",
        "sell_price",
    ],
    "sales_validation": [
        "id",
        "item_id",
        "dept_id",
        "cat_id",
        "store_id",
        "state_id",
    ],
    "sales_evaluation": [
        "id",
        "item_id",
        "dept_id",
        "cat_id",
        "store_id",
        "state_id",
    ],
    "submission": [
        "id",
    ],
    "fred": [
        "date",
        "value",
        "series_id",
    ],
    "holidays": [
        "date",
        "holiday_name",
    ],
    "weather": [
        "time",
        "temperature_2m_max",
        "temperature_2m_min",
        "precipitation_sum",
        "snowfall_sum",
        "wind_speed_10m_max",
        "state_id",
    ],
}

for name, columns in required_columns.items():

    missing_columns = [
        col
        for col in columns
        if col not in datasets[name].columns
    ]

    if not missing_columns:
        print(f"{name.upper():18} PASS")
    else:
        print(
            f"{name.upper():18} "
            f"Missing columns: {missing_columns}"
        )



CALENDAR           PASS
PRICES             PASS
SALES_VALIDATION   PASS
SALES_EVALUATION   PASS
SUBMISSION         PASS
FRED               PASS
HOLIDAYS           PASS
WEATHER            PASS


## 10. Memory Usage

Memory profiling helps determine which datasets need memory-efficient transformations.

`deep=True` includes memory used by object/string values, giving a more realistic RAM estimate.


In [19]:
for name, df in datasets.items():
    memory_mb = df.memory_usage(deep=True).sum() / 1024**2

    print(
        f"{name.upper():18} "
        f"Rows: {df.shape[0]:,} | "
        f"Columns: {df.shape[1]:,} | "
        f"Memory: {memory_mb:,.2f} MB"
    )


CALENDAR           Rows: 1,969 | Columns: 14 | Memory: 0.26 MB
PRICES             Rows: 6,841,121 | Columns: 4 | Memory: 318.15 MB
SALES_VALIDATION   Rows: 30,490 | Columns: 1,919 | Memory: 448.23 MB
SALES_EVALUATION   Rows: 30,490 | Columns: 1,947 | Memory: 454.74 MB
SUBMISSION         Rows: 60,980 | Columns: 29 | Memory: 15.16 MB
FRED               Rows: 198 | Columns: 3 | Memory: 0.01 MB
HOLIDAYS           Rows: 65 | Columns: 2 | Memory: 0.00 MB
WEATHER            Rows: 5,907 | Columns: 7 | Memory: 0.38 MB


### Memory Finding

The M5 sales validation and evaluation tables are the largest wide in-memory objects at roughly **450 MB each** before wide-to-long expansion.

This confirms that the sales transformation should be designed with memory efficiency in mind.


## 11. Missing Value Analysis

Missing values are inspected **before transformation** so they are not removed blindly.

A null can represent either:

- a genuine data-quality problem, or
- a valid business meaning.

For example, the M5 calendar event fields are null on dates when no special event occurs.


In [20]:
for name, df in datasets.items():
    missing = df.isnull().sum()
    missing = missing[missing > 0]

    print(f"\n{name.upper()}")

    if missing.empty:
        print("No missing values.")
    else:
        print(missing)



CALENDAR
event_name_1    1807
event_type_1    1807
event_name_2    1964
event_type_2    1964
dtype: int64

PRICES
No missing values.

SALES_VALIDATION
No missing values.

SALES_EVALUATION
No missing values.

SUBMISSION
No missing values.

FRED
No missing values.

HOLIDAYS
No missing values.

WEATHER
No missing values.


### Missing-Value Interpretation

For the M5 files, missing values in `event_name_1`, `event_type_1`, `event_name_2`, and `event_type_2` are expected because most dates do not correspond to special events.

Those values should **not** be removed simply because they are null.

Any missing values found in FRED, holidays, or weather should be investigated according to the meaning of the affected field before a Transform rule is added.


## 12. Exact Duplicate Analysis

This check looks for rows that are completely identical across every column.

If no exact duplicates exist, there is no reason to add a generic `drop_duplicates()` step simply for the sake of cleaning.


In [21]:
for name, df in datasets.items():
    duplicate_count = df.duplicated().sum()
    print(f"{name.upper():18} Exact duplicate rows: {duplicate_count:,}")


CALENDAR           Exact duplicate rows: 0
PRICES             Exact duplicate rows: 0
SALES_VALIDATION   Exact duplicate rows: 0
SALES_EVALUATION   Exact duplicate rows: 0
SUBMISSION         Exact duplicate rows: 0
FRED               Exact duplicate rows: 0
HOLIDAYS           Exact duplicate rows: 0
WEATHER            Exact duplicate rows: 0


## 13. Business-Key Duplicate Checks

Exact duplicates are not the only type of duplication.

A **business key** represents the columns that should uniquely identify one logical record.


In [22]:
# M5 business keys
price_key_duplicates = prices.duplicated(
    subset=["store_id", "item_id", "wm_yr_wk"]
).sum()

calendar_day_duplicates = calendar["d"].duplicated().sum()
calendar_date_duplicates = calendar["date"].duplicated().sum()

sales_id_duplicates = sales["id"].duplicated().sum()
evaluation_id_duplicates = evaluation["id"].duplicated().sum()
submission_id_duplicates = submission["id"].duplicated().sum()

print("Price store-item-week duplicates:", f"{price_key_duplicates:,}")
print("Calendar d duplicates:", f"{calendar_day_duplicates:,}")
print("Calendar date duplicates:", f"{calendar_date_duplicates:,}")
print("Sales validation ID duplicates:", f"{sales_id_duplicates:,}")
print("Sales evaluation ID duplicates:", f"{evaluation_id_duplicates:,}")
print("Submission ID duplicates:", f"{submission_id_duplicates:,}")

# FRED: one observation per date and series should be unique.
if {"date", "series_id"}.issubset(fred.columns):
    fred_key_duplicates = fred.duplicated(
        subset=["date", "series_id"]
    ).sum()
    print("FRED date-series duplicates:", f"{fred_key_duplicates:,}")


Price store-item-week duplicates: 0
Calendar d duplicates: 0
Calendar date duplicates: 0
Sales validation ID duplicates: 0
Sales evaluation ID duplicates: 0
Submission ID duplicates: 0
FRED date-series duplicates: 0


In [23]:
holiday_key_duplicates = holidays.duplicated(
    subset=["date", "holiday_name"]
).sum()

weather_key_duplicates = weather.duplicated(
    subset=["time", "state_id"]
).sum()

print(
    "Holiday date-name duplicates:",
    f"{holiday_key_duplicates:,}"
)

print(
    "Weather time-state duplicates:",
    f"{weather_key_duplicates:,}"
)

Holiday date-name duplicates: 0
Weather time-state duplicates: 0


### External Business Keys

The external datasets were checked using business keys based on their extracted schemas:

- **Holidays:** one logical record per `date + holiday_name`
- **Weather:** one logical record per `time + state_id`

Both checks returned **0 duplicate business keys**, so no duplicate-removal transformation is required for these datasets.

## 14. Identifier Cardinality

Unique identifier counts help define the primary dimensions of the Walmart data.


In [24]:
identifier_counts = {
    "Stores": sales["store_id"].nunique(),
    "Products": sales["item_id"].nunique(),
    "Departments": sales["dept_id"].nunique(),
    "Categories": sales["cat_id"].nunique(),
    "States": sales["state_id"].nunique(),
}

for name, count in identifier_counts.items():
    print(f"{name}: {count:,}")

if "series_id" in fred.columns:
    print("FRED series:", fred["series_id"].nunique())


Stores: 10
Products: 3,049
Departments: 7
Categories: 3
States: 3
FRED series: 3


### Identifier Finding

The M5 dataset contains:

- **10 stores**
- **3,049 products**
- **7 departments**
- **3 categories**
- **3 states**

These identifiers become important warehouse dimensions and later forecasting features.


## 15. Date Coverage

Date coverage is checked to make sure each source covers the periods required by the forecasting pipeline.

The M5 calendar provides the mapping between `d_*` sales columns and real calendar dates.


In [25]:
calendar_dates = pd.to_datetime(
    calendar["date"],
    errors="coerce",
)

validation_day_columns = [
    col for col in sales.columns
    if col.startswith("d_")
]

evaluation_day_columns = [
    col for col in evaluation.columns
    if col.startswith("d_")
]

print("M5 CALENDAR")
print("Start:", calendar_dates.min())
print("End:  ", calendar_dates.max())
print("Invalid parsed dates:", calendar_dates.isna().sum())
print()
print("Validation day columns:", len(validation_day_columns))
print("Evaluation day columns:", len(evaluation_day_columns))
print("Validation range:", validation_day_columns[0], "to", validation_day_columns[-1])
print("Evaluation range:", evaluation_day_columns[0], "to", evaluation_day_columns[-1])


M5 CALENDAR
Start: 2011-01-29 00:00:00
End:   2016-06-19 00:00:00
Invalid parsed dates: 0

Validation day columns: 1913
Evaluation day columns: 1941
Validation range: d_1 to d_1913
Evaluation range: d_1 to d_1941


### External Date Coverage

The following helper checks every external column containing the word `date`.

`errors="coerce"` converts invalid/unparseable values to `NaT`, allowing the notebook to count bad dates without changing the original DataFrame.


In [26]:
external_date_columns = {
    "fred": "date",
    "holidays": "date",
    "weather": "time",
}

for name, date_column in external_date_columns.items():

    df = datasets[name]

    parsed = pd.to_datetime(
        df[date_column],
        errors="coerce",
    )

    print(f"\n{name.upper()}")

    print(
        f"{date_column}: "
        f"{parsed.min()} -> {parsed.max()} | "
        f"Invalid: {parsed.isna().sum():,}"
    )


FRED
date: 2011-01-01 00:00:00 -> 2016-06-01 00:00:00 | Invalid: 0

HOLIDAYS
date: 2011-01-01 00:00:00 -> 2016-12-26 00:00:00 | Invalid: 0

WEATHER
time: 2011-01-29 00:00:00 -> 2016-06-19 00:00:00 | Invalid: 0


### M5 Date Finding

The M5 calendar spans **2011-01-29 through 2016-06-19** with no invalid parsed dates.

The validation sales file covers `d_1` through `d_1913`, while the evaluation file extends through `d_1941`.

The external date ranges should be compared with the modeling period to confirm that weather, holiday, and economic data provide sufficient historical coverage.


## 16. Basic Value Validity Checks

These checks look for clearly invalid M5 and FRED values that would require investigation or transformation.

The purpose is not to perform deep statistical analysis yet.


In [27]:
print("M5 PRICES / SALES")
print("Missing sell prices:", prices["sell_price"].isna().sum())
print("Zero or negative sell prices:", (prices["sell_price"] <= 0).sum())

sales_values = sales[validation_day_columns]

print("Missing sales values:", sales_values.isna().sum().sum())
print("Negative sales values:", (sales_values < 0).sum().sum())

print("\nFRED")
if "value" in fred.columns:
    fred_numeric = pd.to_numeric(fred["value"], errors="coerce")
    print("Rows:", len(fred))
    print("Values not parseable as numeric:", fred_numeric.isna().sum())


M5 PRICES / SALES
Missing sell prices: 0
Zero or negative sell prices: 0
Missing sales values: 0
Negative sales values: 0

FRED
Rows: 198
Values not parseable as numeric: 0


### Numeric Summary for External Sources

A generic numeric summary is useful for spotting impossible values without assuming a specific weather schema.


In [28]:
for name in ["fred", "weather"]:
    numeric_df = datasets[name].select_dtypes(include="number")

    print(f"\n{name.upper()}")

    if numeric_df.empty:
        print("No numeric columns detected.")
    else:
        display(numeric_df.describe().T)



FRED


,count,mean,std,min,25%,50%,75%,max
value,198.0,79.970303,108.429342,0.07,0.15,7.2,228.68225,240.222



WEATHER


,count,mean,std,min,25%,50%,75%,max
temperature_2m_max,5907.0,20.453106,11.008402,-17.6,13.70,22.3,28.6,42.40
temperature_2m_min,5907.0,11.074742,9.402708,-25.3,5.35,11.7,17.8,31.70
precipitation_sum,5907.0,1.829880,5.981060,0.0,0.00,0.0,0.4,85.50
snowfall_sum,5907.0,0.080037,0.634072,0.0,0.00,0.0,0.0,14.63
wind_speed_10m_max,5907.0,18.474048,7.144428,4.7,13.00,16.7,23.4,48.50


## 17. Dataset Relationships

The source files connect through several identifiers and dates.

### Walmart M5

- `item_id` identifies individual products.
- `store_id` identifies individual stores.
- `wm_yr_wk` connects weekly sell prices to the M5 calendar.
- `d` connects daily sales columns such as `d_1`, `d_2`, ... to actual calendar dates.
- `dept_id` and `cat_id` define the product hierarchy.
- `state_id` connects stores to their states.

### External Data

- FRED economic observations connect to the modeling timeline through `date`.
- Holiday data connects to the calendar through its date field.
- Weather data connects to the modeling timeline through `time` and connects to Walmart stores through `state_id`.

These relationships determine how the Transform and Load layers align the separate sources in PostgreSQL.


## 18. ETL Transformation Requirements

The Transform layer should contain only logic justified by the profiling results.

### Calendar

- Convert `date` from text to datetime.
- Preserve null event fields because they represent dates without special events.
- Retain `d` and `wm_yr_wk` for downstream joins.

### Sell Prices

- Retain `store_id`, `item_id`, `wm_yr_wk`, and `sell_price`.
- Keep `sell_price` numeric.
- Enforce one logical price record per store-item-week.
- Do not add generic null or duplicate removal unless profiling or future validation identifies a real issue.

### Sales

- Preserve the product/store hierarchy identifiers.
- Convert the wide `d_1`, `d_2`, ... columns into long format.
- Join `d` to the calendar to obtain actual dates.
- Keep sales numeric and non-negative.
- Preserve one logical sales series per `id`.

### FRED Economic Data

- Preserve `date`, `value`, and `series_id`.
- Convert `date` to datetime.
- Convert `value` to numeric while handling any source-specific missing-value representation correctly.
- Preserve one observation per `date + series_id`.
- Align economic dates to the forecasting timeline during downstream transformation/feature preparation.

### U.S. Holidays

- Convert the holiday date field to datetime.
- Preserve the holiday/event identity rather than treating a non-holiday day as corrupted data.
- Determine the correct date/event business key from the extracted schema.
- Prepare the holiday data for joining to the calendar/modeling timeline.

### Weather History

- Convert `time` from text to datetime.
- Use `time + state_id` as the expected weather business key.
- Preserve the location/state identifier needed to align weather with Walmart stores.
- Keep weather measurements numeric.
- Investigate missing weather measurements before deciding whether to impute, keep, or reject them.
- Enforce the appropriate date-location business key once the extracted schema is confirmed.

### Sample Submission

- Keep the file as a reference for the M5 28-day forecast output structure.
- Do not treat it as a production warehouse source table.

### Data Quality

- Do not blindly call `dropna()` or `drop_duplicates()`.
- Add a transformation only when profiling shows a justified reason.
- Run validation scripts after transformation/loading so future source-data changes are detected automatically.


## 19. Profiling Findings

The current profiling workflow establishes the following known M5 facts:

- The M5 dataset contains **3,049 products across 10 stores and 3 states**.
- Calendar contains **1,969 rows and 14 columns**.
- Sell prices contains **6,841,121 rows and 4 columns**.
- Sales validation contains **30,490 rows and 1,919 columns**.
- Sales evaluation contains **30,490 rows and 1,947 columns**.
- The validation sales history runs from `d_1` through `d_1913`.
- The evaluation history extends through `d_1941`.
- The M5 calendar spans **2011-01-29 through 2016-06-19**.
- Calendar event nulls are expected rather than corrupted records.
- The sales files require a wide-to-long transformation.
- The calendar `date` field requires datetime conversion.
- No missing, zero, or negative sell prices were found in the current M5 files.
- No missing or negative sales values were found in the validation sales file.
- FRED, holiday, and weather extraction outputs are now included in the same profiling workflow rather than being treated as future placeholders.

The outputs of the external profiling cells should be used to finalize their exact Transform and validation rules.


## 20. Next Steps

After running this entire notebook from top to bottom:

1. Review the profiling results for **all eight datasets**.
2. Record any real external-data issues discovered in FRED, holidays, or weather.
3. Update the corresponding `transform_*.py` logic only where the profiling results justify a transformation.
4. Run the Load scripts to populate PostgreSQL.
5. Run the `validate_*.py` checks against the transformed/loaded data.
6. Perform deeper exploratory data analysis using the analysis-ready data.
7. Engineer forecasting features such as lagged sales, rolling averages, price changes, holidays, weather, and economic indicators.
8. Train and evaluate the demand forecasting models.
9. Use those forecasts to support the dynamic pricing system.
